# Centrality Measures

In this notebook, we explore **centrality measures**, which are used to quantify the importance or influence of individual nodes in a network. A range of centrality measures exists, each offering different perspectives on node importance. The choice of an appropriate measure depends largely on the specific analytical task at hand and the nature of the network being studied. 

NetworkX provides implementations of many centrality measures. For detailed documentation, see:

https://networkx.org/documentation/stable/reference/algorithms/centrality.html

In [1]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.options.display.float_format = '{:.3f}'.format

First, let's load a small sample network representing connections on the LinkedIn platform, which is stored in GEXF format.

In [2]:
g = nx.read_gexf("linkedin25.gexf")

FileNotFoundError: [Errno 2] No such file or directory: 'linkedin25.gexf'

We see this is an undirected network:

In [ ]:
g.is_directed()

Produce a simple visualisation of the network:

In [ ]:
plt.figure(figsize=(10,8))
nx.draw_networkx(g, 
                 with_labels=True, 
                 node_size=800, 
                 node_color="lightblue")
plt.axis("off")
plt.show()

## Node Degree

The **degree** of a node is the most basic measure of centrality. It refers to the number of nodes to which it is directly connected in the network. In other words, a node's degree equals its number of immediate neighbours. The `degree()` method associated with a network returns the degree for all nodes in a network, or for a specified individual node:

In [ ]:
# get degree score for a single user
g.degree("Tina Davis")

In [ ]:
# get a dictionary of degree scores for all nodes
degrees = dict(g.degree())
degrees

We can examine various statistical properties of the degree sequence in this network to better understand its structural characteristics:

In [ ]:
# turn the values into a Pandas Series first
degree_seq = pd.Series(degrees)
degree_seq

In [ ]:
# now calculated various statistics for the Series
print(f"Degree range: [{degree_seq.min()}, {degree_seq.max()}]")
print(f"Mean degree: {degree_seq.mean():.2f}")
print(f"Median degree: {degree_seq.median():.0f}")

We can also generate a visualisation of the **degree distribution** for this network, which reveals how connectivity is distributed amongst the nodes:

In [ ]:
ax = degree_seq.plot.hist(figsize=(7, 6), 
                          fontsize=12, 
                          legend=None, 
                          color="darkred")
ax.set_ylabel("Number of Nodes", fontsize=12)
ax.set_xlabel("Degree", fontsize=12)
plt.show()

### Measuring Centrality

Centrality analysis allows us to identify the most important nodes in a network. The actual definition of importance depends on the nature of the network, and many different centrality measures exist. NetworkX includes implementations of the most common measures.

The most basic measure is **degree centrality**, which normalises the raw degree of each node by dividing it by $(n-1)$, where $n$ is the total number of nodes in the network. This normalisation ensures that degree centrality scores range between 0 and 1, allowing comparisons across different networks. The `nx.degree_centrality()` function returns a dictionary where the keys represent the nodes and the values contain their corresponding centrality scores.

In [ ]:
# calculate the degree centrality for every node
deg = nx.degree_centrality(g)
# iterate over the resulting dictionary
for node, centrality in deg.items():
    print(f"{node}: {centrality:.3f}")

We can use these scores to create a Pandas DataFrame and display a ranking of the nodes by their degree centrality:

In [ ]:
s = pd.Series(deg)
# turn the scores into a DataFrame
df = pd.DataFrame(s, columns=["degree_centrality"])
# display the DataFrame sorted by degree centrality
df.sort_values(by="degree_centrality", ascending=False).head(10)

Another important measure, **betweenness centrality**, is useful for identifying "broker" or "bridging" nodes within a network structure. This measure quantifies how frequently a node appears on the shortest paths connecting other pairs of nodes in the network. Nodes with high betweenness centrality scores often serve as crucial intermediaries, controlling the flow of information or resources between different parts of the network.

In [ ]:
bet = nx.betweenness_centrality(g)
# iterate over the resulting dictionary
for node, centrality in bet.items():
    print(f"{node}: {centrality:.3f}")

In [ ]:
# alternatively, add this as a column in our existing DataFrame
df["betweenness"] = pd.Series(bet)
df.sort_values(by="betweenness", ascending=False).head(10)

**Closeness centrality** measures the extent to which a node maintains close proximity to all other nodes in a network, considering both direct and indirect connections. This measure reflects how quickly a node can reach other nodes in the network, making it particularly relevant for understanding efficiency of communication or influence propagation.

In [ ]:
close = nx.closeness_centrality(g)
# iterate over the resulting dictionary
for node, centrality in close.items():
    print(f"{node}: {centrality:.3f}")

In [ ]:
# alternatively, add this as a column in our existing DataFrame
df["closeness"] = pd.Series(close)
df.sort_values(by="closeness", ascending=False).head(10)

The **eigenvector centrality** of a node is calculated as being proportional to the sum of the centrality scores of its immediate neighbours. This creates a recursive definition where a node is considered important if it is connected to other important nodes. This measure captures the concept that being connected to highly central nodes is more valuable than being connected to peripheral ones.

In [ ]:
eig = nx.eigenvector_centrality(g)
# just add this as a column in our existing DataFrame
df["eigenvector"] = pd.Series(eig)
df.sort_values(by="eigenvector", ascending=False).head(10)

To allow comparisons to be made across different networks, we can apply normalisation relative to the maximum value observed in the current network, ensuring that the highest-scoring node receives a value of 1.0.

In [ ]:
# add this as a column in our existing DataFrame
df["norm_eigenvector"] = df["eigenvector"]/max(df["eigenvector"])
df.sort_values(by="norm_eigenvector", ascending=False).head(10)

As we can see from the DataFrame, the ranking order produced by the various centrality measures can differ substantially, particularly in the case of betweenness centrality. This shows that different measures capture fundamentally different aspects of node importance.

We can quantify these differences by examining the correlation coefficients between the different measures (i.e., the columns of the DataFrame):

In [ ]:
df.corr()